## 1. Problem Formulation



### Decision Variables
Let $X_{i,j,m}$ represent the number of units shipped from fulfillment center $i$ to region $j$ using transportation method $m$.
* $i \in \{Boston, Denver, LA\}$
* $j \in \{East, Central, West\}$
* $m \in \{Ground, Air\}$

*Note: Variables are only created for routes where the delivery time is less than or equal to the region's delivery deadline.*

### Objective Function
Minimize the total transportation costs:
$$\text{Minimize } Z = \sum_{i,j,m} C_{i,j,m} \cdot X_{i,j,m}$$
Where $C_{i,j,m}$ is the cost per unit for a given route and method.

### Constraints
1.  **Stock Limits (Supply):** The total units shipped from each center cannot exceed its available inventory.
    * $\sum_{j,m} X_{\text{Boston},j,m} \le 500$
    * $\sum_{j,m} X_{\text{Denver},j,m} \le 400$
    * $\sum_{j,m} X_{\text{LA},j,m} \le 300$
2.  **Demand Satisfaction:** Each region must receive exactly its requested demand.
    * $\sum_{i,m} X_{i,\text{East},m} = 450$
    * $\sum_{i,m} X_{i,\text{Central},m} = 350$
    * $\sum_{i,m} X_{i,\text{West},m} = 300$
3.  **Non-Negativity:** * $X_{i,j,m} \ge 0$

In [2]:
import pulp

def solve_festiveco_network(demand_adjustments=None, deadline_adjustments=None):
    """
    Modular function to solve the FestiveCo distribution problem.
    Allows for dictionary inputs to adjust demands or deadlines for What-If analysis.
    """
    # --- 1. Define Data ---
    supply = {'Boston': 500, 'Denver': 400, 'LA': 300}
    
    demand = {'East': 450, 'Central': 350, 'West': 300}
    if demand_adjustments:
        demand.update(demand_adjustments)
        
    deadlines = {'East': 5, 'Central': 10, 'West': 15}
    if deadline_adjustments:
        deadlines.update(deadline_adjustments)
    
    # Format: (Origin, Destination, Method): (Cost, Days)
    routes_data = {
        ('Boston', 'East', 'Ground'): (10, 3),
        ('Boston', 'East', 'Air'): (25, 1),
        ('Denver', 'Central', 'Ground'): (15, 5),
        ('Denver', 'Central', 'Air'): (35, 2),
        ('LA', 'West', 'Ground'): (20, 7),
        ('LA', 'West', 'Air'): (50, 3),
        ('Boston', 'Central', 'Ground'): (18, 6),
        ('Boston', 'Central', 'Air'): (40, 2),
        ('Denver', 'East', 'Ground'): (22, 8),
        ('Denver', 'East', 'Air'): (45, 3),
        ('LA', 'East', 'Ground'): (30, 10),
        ('LA', 'East', 'Air'): (60, 4)
    }

    # Filter out routes that violate delivery deadlines
    valid_routes = {
        route: data for route, data in routes_data.items() 
        if data[1] <= deadlines[route[1]]
    }

    # --- 2. Initialize Model ---
    prob = pulp.LpProblem("FestiveCo_Distribution", pulp.LpMinimize)

    # --- 3. Decision Variables ---
    # Create variables only for valid routes
    route_vars = pulp.LpVariable.dicts("Route", valid_routes.keys(), lowBound=0, cat='Continuous')

    # --- 4. Objective Function ---
    prob += pulp.lpSum([route_vars[r] * valid_routes[r][0] for r in valid_routes]), "Total_Cost"

    # --- 5. Constraints ---
    # Supply constraints
    for origin in supply.keys():
        prob += pulp.lpSum([route_vars[r] for r in valid_routes if r[0] == origin]) <= supply[origin], f"Supply_{origin}"

    # Demand constraints
    for dest in demand.keys():
        prob += pulp.lpSum([route_vars[r] for r in valid_routes if r[1] == dest]) == demand[dest], f"Demand_{dest}"

    # --- 6. Solve and Return Results ---
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    status = pulp.LpStatus[prob.status]
    total_cost = pulp.value(prob.objective)
    solution = {r: route_vars[r].varValue for r in valid_routes if route_vars[r].varValue > 0}
    
    return status, total_cost, solution

# Execute Base Scenario
status, total_cost, solution = solve_festiveco_network()

print(f"Base Scenario Status: {status}")
print(f"Optimal Total Cost: ${total_cost:,.2f}\n")
print("Optimal Shipping Routes:")
for route, qty in solution.items():
    print(f" - Ship {int(qty)} units from {route[0]} to {route[1]} via {route[2]}")

Base Scenario Status: Optimal
Optimal Total Cost: $15,750.00

Optimal Shipping Routes:
 - Ship 450 units from Boston to East via Ground
 - Ship 350 units from Denver to Central via Ground
 - Ship 300 units from LA to West via Ground


## Analysis of the Base Solution
* Optimal Total Cost: $15,750.00
* Routing Strategy:
    * Boston to East (Ground): 450 units. (Utilizes the cheapest $10 route, arriving in 3 days, easily beating the 5-day deadline).
    * Denver to Central (Ground): 350 units. (Utilizes the cheapest $15 route, arriving in 5 days to beat the 10-day deadline).
    * LA to West (Ground): 300 units. (Utilizes the $20 route, arriving in 7 days to beat the 15-day deadline).

The model successfully avoids all expensive air shipping because the ground shipping times comfortably meet the deadlines for these optimal, geographically sensible pairings.

In [3]:
# What-If Scenario 1: East deadline extended to 10 days
print("--- What-If Scenario 1: East Deadline Extended to 10 Days ---")
status_1, cost_1, sol_1 = solve_festiveco_network(deadline_adjustments={'East': 10})
print(f"Status: {status_1} | Total Cost: ${cost_1:,.2f}")
for route, qty in sol_1.items():
    print(f" - {int(qty)} units: {route[0]} -> {route[1]} ({route[2]})")

print("\n--- What-If Scenario 2: Central Demand Increases by 50 units ---")
# What-If Scenario 2: Central demand is now 400 (350 + 50)
status_2, cost_2, sol_2 = solve_festiveco_network(demand_adjustments={'Central': 400})
print(f"Status: {status_2} | Total Cost: ${cost_2:,.2f}")
for route, qty in sol_2.items():
    print(f" - {int(qty)} units: {route[0]} -> {route[1]} ({route[2]})")

--- What-If Scenario 1: East Deadline Extended to 10 Days ---
Status: Optimal | Total Cost: $15,750.00
 - 450 units: Boston -> East (Ground)
 - 350 units: Denver -> Central (Ground)
 - 300 units: LA -> West (Ground)

--- What-If Scenario 2: Central Demand Increases by 50 units ---
Status: Optimal | Total Cost: $16,500.00
 - 450 units: Boston -> East (Ground)
 - 400 units: Denver -> Central (Ground)
 - 300 units: LA -> West (Ground)


## What-If Analysis Results

### 1. East Deadline Extended to 10 Days
* **Effect on Solution:** None. 
* **Total Cost:** $15,750.00 (Remains the same).
* **Reasoning:** Extending the deadline makes the Denver-East (Ground) and LA-East (Ground) routes valid. However, the Boston-East (Ground) route at $10/unit remains the absolute cheapest option across the entire network. Since Boston has enough capacity (500) to fulfill the East's demand (450), the optimizer sticks with the original plan.

### 2. Central Demand Increases by 50 Units (to 400)
* **Effect on Solution:** Feasible. Total cost increases to $16,500.00.
* **Reasoning:** Total network supply is 1,200 units, and the new total demand is 1,150 units, so the system can handle the capacity. Denver originally sent 350 units to Central and had a maximum capacity of 400. The model efficiently utilizes Denver's exact remaining 50 units of capacity to fulfill Central's increased demand via the existing $15/unit Ground route.
    * *Cost Math:* Original $15,750 + (50 extra units * $15/unit) = $16,500.